# CNN Practice: CIFAR-10-Style Image Classification

This notebook demonstrates two convolutional neural networks using synthetic image data shaped like CIFAR-10 images. The data is randomly generated for practice, so the accuracy is not meaningful for real image classification.

We compare:
1. A compact baseline CNN.
2. A deeper CNN with an additional convolutional block.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    GlobalAveragePooling2D, Dense, Dropout
)

SEED = 21
rng = np.random.default_rng(SEED)
tf.keras.utils.set_random_seed(SEED)

IMAGE_SHAPE = (32, 32, 3)
CLASS_COUNT = 10
SAMPLE_COUNT = 1200

images = rng.random((SAMPLE_COUNT, *IMAGE_SHAPE), dtype=np.float32)
targets = rng.integers(0, CLASS_COUNT, size=SAMPLE_COUNT)
targets = tf.keras.utils.to_categorical(targets, CLASS_COUNT)

print("Images:", images.shape)
print("Targets:", targets.shape)


## 1. Baseline convolutional network

In [ ]:
baseline = Sequential([
    Input(shape=IMAGE_SHAPE),
    Conv2D(24, 3, padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=2),
    GlobalAveragePooling2D(),
    Dropout(0.25),
    Dense(48, activation="relu"),
    Dense(CLASS_COUNT, activation="softmax")
], name="baseline_cnn")

baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

baseline.summary()


In [ ]:
baseline_history = baseline.fit(
    images,
    targets,
    epochs=4,
    batch_size=40,
    validation_split=0.2,
    verbose=1
)


## 2. Deeper CNN with an extra feature-extraction block

In [ ]:
deeper_cnn = Sequential([
    Input(shape=IMAGE_SHAPE),

    Conv2D(32, 3, padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=2),

    Conv2D(64, 3, padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(pool_size=2),

    GlobalAveragePooling2D(),
    Dropout(0.35),
    Dense(80, activation="relu"),
    Dropout(0.20),
    Dense(CLASS_COUNT, activation="softmax")
], name="deeper_cnn")

deeper_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0008),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

deeper_cnn.summary()


In [ ]:
deeper_history = deeper_cnn.fit(
    images,
    targets,
    epochs=4,
    batch_size=40,
    validation_split=0.2,
    verbose=1
)


## Notes

- `Conv2D` learns local visual patterns.
- `BatchNormalization` stabilizes activations during training.
- `MaxPooling2D` reduces the spatial size of feature maps.
- `GlobalAveragePooling2D` converts feature maps into a compact vector.
- `Dropout` randomly disables some units during training to reduce overfitting.
- Because the inputs and labels are random, this notebook is intended for API/model practice rather than measuring image-classification performance.
